# 🛡️ Step 3: Unity Catalog Governance

This notebook demonstrates Unity Catalog governance features:
1. **Column-level tags** — Mark sensitive columns (PII, threat_intel, etc.)
2. **Table comments** — Document all tables for discoverability
3. **Row-level security** — SOC analysts see only their assigned customers
4. **Data lineage** — Verify lineage from Bronze → Silver → Gold

In [0]:
%run ./01_config

# ⚙️ Configuration — SOC Alert Triage

Shared configuration for all notebooks. Update the catalog name below before running.

✅ Configuration loaded
   Catalog: rajarora_febar
   Schema:  soc_alert_triage
   Tables:  10 tables configured


In [0]:
table_comments = {
    "raw_security_events": "Bronze layer: Raw security telemetry from MDR sensors, ingested from landing files by Auto Loader. Includes firewall, EDR, auth, email, and cloud audit events.",
    "threat_intel_iocs": "Reference table: Known-bad indicators of compromise (IOCs) from multiple threat intelligence feeds.",
    "asset_inventory": "Reference table: Managed assets (servers, endpoints, network devices) across all MDR customers.",
    "customers": "Reference table: MDR customer accounts with industry, tier, and region metadata.",
    "silver_normalized_events": "Silver layer: Parsed, deduplicated, and OCSF-normalized security events with extracted fields.",
    "gold_enriched_alerts": "Gold layer: Enriched security alerts with threat intel correlation, asset context, and composite risk scores.",
    "historical_triage_decisions": "Training data: Historical analyst triage decisions used for ML model training.",
}

for table_short, comment in table_comments.items():
    full_table = f"{FULL_SCHEMA}.{table_short}"
    try:
        spark.sql(f"COMMENT ON TABLE {full_table} IS '{comment}'")
        print(f"✅ Comment added to {table_short}")
    except Exception as e:
        print(f"⚠️ Could not comment {table_short}: {e}")

✅ Comment added to raw_security_events
✅ Comment added to threat_intel_iocs
✅ Comment added to asset_inventory
✅ Comment added to customers
✅ Comment added to silver_normalized_events
✅ Comment added to gold_enriched_alerts
✅ Comment added to historical_triage_decisions


In [0]:
# Allowed tag values on this workspace: confidential, restricted, public, internal
tags = [
    # Customer PII → confidential
    (f"{FULL_SCHEMA}.customers", "customer_name", "confidential"),
    # Network identifiers → restricted
    (f"{FULL_SCHEMA}.silver_normalized_events", "src_ip", "restricted"),
    (f"{FULL_SCHEMA}.silver_normalized_events", "dst_ip", "restricted"),
    (f"{FULL_SCHEMA}.silver_normalized_events", "username", "confidential"),
    # Threat intelligence → restricted
    (f"{FULL_SCHEMA}.threat_intel_iocs", "ioc_value", "restricted"),
    (f"{FULL_SCHEMA}.gold_enriched_alerts", "threat_intel_match", "restricted"),
    (f"{FULL_SCHEMA}.gold_enriched_alerts", "composite_risk_score", "internal"),
    (f"{FULL_SCHEMA}.gold_enriched_alerts", "risk_tier", "internal"),
    # Asset info → restricted
    (f"{FULL_SCHEMA}.asset_inventory", "ip_address", "restricted"),
    (f"{FULL_SCHEMA}.asset_inventory", "hostname", "restricted"),
]

for table, column, tag in tags:
    try:
        spark.sql(f"ALTER TABLE {table} ALTER COLUMN {column} SET TAGS ('classification' = '{tag}')")
        print(f"✅ Tagged {table.split('.')[-1]}.{column} as '{tag}'")
    except Exception as e:
        print(f"⚠️ Could not tag {column}: {e}")

✅ Tagged customers.customer_name as 'confidential'
✅ Tagged silver_normalized_events.src_ip as 'restricted'
✅ Tagged silver_normalized_events.dst_ip as 'restricted'
✅ Tagged silver_normalized_events.username as 'confidential'
✅ Tagged threat_intel_iocs.ioc_value as 'restricted'
✅ Tagged gold_enriched_alerts.threat_intel_match as 'restricted'
✅ Tagged gold_enriched_alerts.composite_risk_score as 'internal'
✅ Tagged gold_enriched_alerts.risk_tier as 'internal'
✅ Tagged asset_inventory.ip_address as 'restricted'
✅ Tagged asset_inventory.hostname as 'restricted'


In [0]:
# Who may see which customers' alerts. customer_id '*' = all customers.
# - The pipeline runner (data engineer) needs all rows to score/summarize every alert
# - The Databricks App's service principal is a "Tier 1 analyst" assigned 10 customers
APP_SERVICE_PRINCIPAL = "da553d37-8a52-4c9e-b8c6-493e4d26addb"  # soc-triage app SP (application ID)

pipeline_user = spark.sql("SELECT current_user()").first()[0]
assigned_customers = [r.customer_id for r in spark.table(TABLES["customers"]).orderBy("customer_id").limit(10).collect()]

assignments = [(pipeline_user, "*", "data_engineer")] + [
    (APP_SERVICE_PRINCIPAL, cid, "tier1_analyst") for cid in assigned_customers
]
(spark.createDataFrame(assignments, "principal STRING, customer_id STRING, role STRING")
    .write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{FULL_SCHEMA}.analyst_customer_assignments"))
spark.sql(f"COMMENT ON TABLE {FULL_SCHEMA}.analyst_customer_assignments IS "
          "'RLS mapping: which principals may see which customers (customer_id * = all)'")
print(f"✅ Assignments: {pipeline_user} → all customers; app SP → {len(assigned_customers)} customers")
spark.table(f"{FULL_SCHEMA}.analyst_customer_assignments").display()

✅ Assignments: rajneesh.arora@databricks.com → all customers; app SP → 10 customers


principal,customer_id,role
rajneesh.arora@databricks.com,*,data_engineer
da553d37-8a52-4c9e-b8c6-493e4d26addb,CUST-0001,tier1_analyst
da553d37-8a52-4c9e-b8c6-493e4d26addb,CUST-0002,tier1_analyst
da553d37-8a52-4c9e-b8c6-493e4d26addb,CUST-0003,tier1_analyst
da553d37-8a52-4c9e-b8c6-493e4d26addb,CUST-0004,tier1_analyst
da553d37-8a52-4c9e-b8c6-493e4d26addb,CUST-0005,tier1_analyst
da553d37-8a52-4c9e-b8c6-493e4d26addb,CUST-0006,tier1_analyst
da553d37-8a52-4c9e-b8c6-493e4d26addb,CUST-0007,tier1_analyst
da553d37-8a52-4c9e-b8c6-493e4d26addb,CUST-0008,tier1_analyst
da553d37-8a52-4c9e-b8c6-493e4d26addb,CUST-0009,tier1_analyst


In [0]:
# SOC managers see all data; everyone else sees only their assigned customers

spark.sql(f"""
CREATE OR REPLACE FUNCTION {FULL_SCHEMA}.rls_customer_filter(cust_id STRING)
RETURNS BOOLEAN
COMMENT 'Row-level security filter: SOC managers see all; others see customers assigned in analyst_customer_assignments'
RETURN
    IS_ACCOUNT_GROUP_MEMBER('soc_managers')
    OR EXISTS (
        SELECT 1 FROM {FULL_SCHEMA}.analyst_customer_assignments
        WHERE principal = current_user() AND customer_id = '*'
    )
    OR cust_id IN (
        SELECT customer_id FROM {FULL_SCHEMA}.analyst_customer_assignments
        WHERE principal = current_user()
    )
""")
print("✅ RLS function created: rls_customer_filter")

✅ RLS function created: rls_customer_filter


In [0]:
# No try/except: if the row filter cannot be applied, this task (and the job) must fail
# rather than leave gold alerts readable by every analyst.
spark.sql(f"""
    ALTER TABLE {FULL_SCHEMA}.gold_enriched_alerts 
    SET ROW FILTER {FULL_SCHEMA}.rls_customer_filter ON (customer_id)
""")
row_filter = spark.sql(f"DESCRIBE TABLE EXTENDED {FULL_SCHEMA}.gold_enriched_alerts").filter("col_name = 'Row Filter'").collect()
assert row_filter, "Row filter is not set on gold_enriched_alerts"
print(f"✅ Row filter applied to gold_enriched_alerts: {row_filter[0].data_type}")

✅ Row filter applied to gold_enriched_alerts: `rajarora_febar`.`soc_alert_triage`.`rls_customer_filter` ON (customer_id)


### Data Lineage Verification

Unity Catalog automatically tracks lineage. You can verify it in the UI:
1. Navigate to **Catalog Explorer** → `rajarora_febar.soc_alert_triage`
2. Click on `gold_enriched_alerts`
3. Go to the **Lineage** tab

You should see:
```
raw_security_events (Bronze)
  → silver_normalized_events (Silver)
    → gold_enriched_alerts (Gold)
      ← threat_intel_iocs (Reference)
      ← asset_inventory (Reference)
      ← customers (Reference)
```

In [0]:
print("=" * 60)
print("🛡️ UNITY CATALOG GOVERNANCE SUMMARY")
print("=" * 60)

# List all tables with their comments
tables_info = spark.sql(f"""
    SELECT table_name, comment 
    FROM {CATALOG_NAME}.information_schema.tables 
    WHERE table_schema = '{SCHEMA_NAME}'
    ORDER BY table_name
""")
tables_info.display()

print("\n✅ Unity Catalog governance configured!")
print("   - Table comments added for discoverability")
print("   - Column-level tags applied (PII, threat_intel, risk_metric)")
print("   - Row-level security function created")
print("   - Data lineage tracked automatically")
print("\nNext step: Run 04_ml_classifier to train the ML model")

🛡️ UNITY CATALOG GOVERNANCE SUMMARY


table_name,comment
alert_queue_synced,Database table synced from another UC table
analyst_actions,"Analyst triage actions — OLTP writes from the Databricks App, synced back to lakehouse for analytics"
analyst_assignments_synced,Database table synced from another UC table
analyst_customer_assignments,RLS mapping: which principals may see which customers (customer_id * = all)
asset_inventory,"Reference table: Managed assets (servers, endpoints, network devices) across all MDR customers."
customers,"Reference table: MDR customer accounts with industry, tier, and region metadata."
event_log_063d309b_31a2_4a92_8b50_38da61f994c2,null
event_log_edeeacf9_84b7_443d_ac5a_62c4bc1c7e85,null
gold_alert_predictions,null
gold_alert_queue,"Serving table: critical/high alerts with ML prediction + AI summary, synced to Lakebase for the analyst app"



✅ Unity Catalog governance configured!
   - Table comments added for discoverability
   - Column-level tags applied (PII, threat_intel, risk_metric)
   - Row-level security function created
   - Data lineage tracked automatically

Next step: Run 04_ml_classifier to train the ML model
